In [1]:
from pyproj import Transformer

lat_tl, lon_tl = 63.143137, 7.642113  # top-left  (NW corner)
lat_br, lon_br = 63.097222, 7.774598  # bottom-right (SE corner)

# WGS84 (EPSG:4326)  ->  UTM zone 32N (EPSG:32632)
# to_utm = Transformer.from_crs(4326, 32632, always_xy=True)

# WGS84 (EPSG:4326)  ->  UTM zone 33N (EPSG:32633)
to_utm = Transformer.from_crs(4326, 32633, always_xy=True)


e_tl, n_tl = to_utm.transform(lon_tl, lat_tl)
e_br, n_br = to_utm.transform(lon_br, lat_br)

east_min,  east_max  = min(e_tl, e_br), max(e_tl, e_br)
north_min, north_max = min(n_tl, n_br), max(n_tl, n_br)

width  = int(east_max  - east_min)
height = int(north_max - north_min)
origin = (int(east_min), int(north_min))   # lower-left corner

Write a minimal `config.yaml` for an FGDB source

Adjust `resources` to point at your actual FGDB directory inside `data/db/`
(e.g. the folder name of the Kartverket chart covering Trondheimsfjorden).

In [2]:
from pathlib import Path
import textwrap

# You can fecth this data from Kartverket's download service:
FGDB_NAME = "../data/db/Basisdata_0000_Norge_25833_Dybdedata_FGDB.gdb"
config_yaml = textwrap.dedent(f"""\
    enc:
      size:   [{width}, {height}]
      origin: [{origin[0]}, {origin[1]}]
      crs:   "UTM33N"
      S57_layers: {{}}
      resources: ["{FGDB_NAME}"]

    display:
      colorbar:   false
      controls:   false
      dark_mode:  false
      grayscale_depths: True
      resolution: 900
      dpi:        96
      binary_sea_land: true
""")

Path("config.yaml").write_text(config_yaml)
print(config_yaml)

enc:
  size:   [6069, 5850]
  origin: [129734, 7016938]
  crs:   "UTM33N"
  S57_layers: {}
  resources: ["../data/db/Basisdata_0000_Norge_25833_Dybdedata_FGDB.gdb"]

display:
  colorbar:   false
  controls:   false
  dark_mode:  false
  grayscale_depths: True
  resolution: 900
  dpi:        96
  binary_sea_land: true



Load the ENC and parse the FGDB data

In [3]:
from seacharts import ENC

enc = ENC("config.yaml")
enc.update()   # parses the FGDB under data/db/<FGDB_NAME> into shapefiles

print("Seabed depth bins available:", sorted(enc.seabed.keys()))

INFO: ENC created using data from existing shapefiles.

INFO: Updating ENC with data from available resources...

Processing 35 km^2 of ENC features:
Found 0 Seabed350m geometries.

ENC update complete.

INFO: Updating ENC with data from available resources...

Processing 35 km^2 of ENC features:
Found 0 Seabed350m geometries.

ENC update complete.

Seabed depth bins available: [0, 1, 2, 5, 10, 20, 50, 100, 200, 350, 500]


`get_depth_at_coord` takes **UTM** easting/northing. The helper below lets you
pass lat/lon directly.

In [4]:
easting, northing = to_utm.transform(10.35, 63.45)

print(f"Depth is {enc.get_depth_at_coord(easting, northing)}")

Depth is None


In [5]:
from matplotlib.colors import to_hex

cbar = enc.display._colorbar
resolution = enc.size[1] / enc.display._resolution

print(f"image: ./base.png")
print(f"negate: 0")
print(f"free_thresh: 0.196")
print(f"occupied_thresh: 0.65")
print(f"resolution: {resolution:.6f}")
print(f"origin: [0.0, 0.0, 0.0]")
print("layers:")
print("  - name: depth")
print("    file: ./depth.png")
print("    values:")
for depth in enc.depth_bins:
    hex_color = to_hex(cbar.cmap(cbar.norm(depth)))
    print(f"      '{hex_color}': {float(depth):.1f}")

image: ./base.png
negate: 0
free_thresh: 0.196
occupied_thresh: 0.65
resolution: 6.500000
origin: [0.0, 0.0, 0.0]
layers:
  - name: depth
    file: ./depth.png
    values:
      '#6abf71': 0.0
      '#4a98c9': 1.0
      '#3e8ec4': 2.0
      '#3383be': 5.0
      '#2979b9': 10.0
      '#1f6eb3': 20.0
      '#1764ab': 50.0
      '#0e59a2': 100.0
      '#084f99': 200.0
      '#083a7a': 350.0
      '#08306b': 500.0


In [6]:
display = enc.display

In [7]:
import numpy as np
from PIL import Image
from pathlib import Path

# 1) Save raw depth render
enc.display.save_image(name="depth", path=Path("report"), extension="png")

# 2) Post-process depth.png into base.png
img = Image.open("report/depth.png").convert("RGBA")
arr = np.array(img, dtype=np.uint8)

r = arr[:, :, 0].astype(np.int16)
g = arr[:, :, 1].astype(np.int16)
b = arr[:, :, 2].astype(np.int16)

# Sea is blue-dominant in this palette.
sea_mask = (b > g + 8) & (b > r + 8)
land_mask = ~sea_mask

# Remove color from land/shore by forcing a neutral grayscale value.
arr[land_mask, :3] = 0

# Make sea transparent.
arr[sea_mask, 3] = 0

Image.fromarray(arr, mode="RGBA").save("report/base.png")
print("Saved ./depth.png")
print(f"land pixels kept (neutral): {int(land_mask.sum())}")
print(f"sea pixels transparent: {int(sea_mask.sum())}")

Saved ./depth.png
land pixels kept (neutral): 246587
sea pixels transparent: 593113


In [8]:
import yaml
from matplotlib.colors import to_hex

from seacharts.display.colors import color_picker

# Build the exact depth-to-color mapping used by the chart layers.
bins = len(enc.depth_bins)
depth_values = {
    to_hex(color_picker(i, bins), keep_alpha=False) : float(depth)
    for i, depth in enumerate(enc.depth_bins)
}

yaml_data = {
    "image": "./base.png",
    "negate": 0,
    "free_thresh": 0.196,
    "occupied_thresh": 0.65,
    "resolution": round(float(resolution), 6),
    "origin": [0.0, 0.0, 0.0],
    "layers": [
        {
            "name": "depth",
            "file": "./depth.png",
            "values": depth_values,
        }
    ],
}

with open("report/map.yaml", "w", encoding="utf-8") as yaml_file:
    yaml.safe_dump(yaml_data, yaml_file, sort_keys=False)

print("Wrote report/map.yaml")
print(depth_values)

Wrote report/map.yaml
{'#4a98c9': 0.0, '#3e8ec4': 1.0, '#3383be': 2.0, '#2979b9': 5.0, '#1f6eb3': 10.0, '#1764ab': 20.0, '#0e59a2': 50.0, '#084f99': 100.0, '#08458a': 200.0, '#083a7a': 350.0, '#08306b': 500.0}
